# Method C: Slow & Steady Validation (v2 — Live Fingerprinting)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kduvekot/ubl-gc/blob/claude/check-revision-diff-8j9aY/notebooks/method-c-slow-validation.ipynb)

## What changed from v1

v1 used **hardcoded `CURRENT_HASHES`** to detect collapsed downloads. This
failed because the API returns new hash variants of the same current content
(ODS metadata/style jitter). The collapse at rev 1710 went undetected because
hash `a2bd37a0...` wasn't in the known set.

v2 replaces this with **live reference fingerprinting**:
1. Download MAX_REV (the latest revision = current content by definition)
2. Extract a **data fingerprint** — only cell text from data tables, ignoring
   all ODS metadata, styles, formulas, and the Logs-sheet
3. For each revision download, compare its fingerprint against the reference
4. No hardcoded hashes. No assumptions about what "current" looks like.

## How to use

1. Set `TEST_SHEET` in the config cell
2. **Run All** — the notebook will:
   - Authenticate and mount Drive
   - Download MAX_REV to establish a live reference fingerprint
   - Audit existing Drive files to find collapsed/missing ones
   - Auto-compute the best resume point
   - Download remaining revisions with adaptive pacing
   - Stop cleanly on canary (10 identical fingerprints in a row)
3. If it stops (canary, timeout, crash), just do **Run All** again — it
   picks up from the checkpoint automatically

In [ ]:
# === Step 0: Auth + Mount ===
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

import google.auth
from google.auth.transport.requests import Request as AuthRequest
creds, project = google.auth.default(
    scopes=['https://www.googleapis.com/auth/drive']
)
creds.refresh(AuthRequest())
TOKEN = creds.token
print(f'Token: {TOKEN[:15]}...{TOKEN[-4:]}')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/ubl-gc-revisions')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Drive: {DRIVE_DIR}')

## Step 1: Configuration

In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │  CONFIGURATION — adjust these for each run                     │
# └─────────────────────────────────────────────────────────────────┘

# Delay between API calls (seconds). Adaptive backoff adjusts this.
REQUEST_DELAY = 10.0

# Stop after this many consecutive identical data fingerprints.
CANARY_THRESHOLD = 10

# ── Batch cooldown: pause every N revisions to let the API "forget" ──
BATCH_SIZE = 500           # download this many, then pause
COOLDOWN_MINUTES = 10      # pause duration between batches

# How many verified-good revisions to re-download before the first
# problem revision (to confirm the API is returning real data).
RESUME_MARGIN = 20

# Refresh the reference fingerprint every N downloads, in case
# someone edits the spreadsheet during the run.
REFERENCE_REFRESH_INTERVAL = 500

# Which sheet to download
TEST_SHEET = 'ubl25_library'
# TEST_SHEET = 'ubl25_documents'

# Max revision number per sheet (from Revisions API discovery).
MAX_REVS = {
    'ubl25_library':   2005,
    'ubl25_documents': 2204,
}

MAX_REV = MAX_REVS[TEST_SHEET]

print(f'Config: {TEST_SHEET}')
print(f'  Revisions: 1 -> {MAX_REV}')
print(f'  Delay: {REQUEST_DELAY}s between requests (adaptive)')
print(f'  Canary: stop after {CANARY_THRESHOLD} identical fingerprints in a row')
print(f'  Batch: {BATCH_SIZE} revisions, then {COOLDOWN_MINUTES}min cooldown')
print(f'  Resume margin: {RESUME_MARGIN} revisions before first problem')
print(f'  Reference refresh: every {REFERENCE_REFRESH_INTERVAL} downloads')

## Step 2: Helpers

In [ ]:
import json, time, hashlib, gzip, zipfile, io, re
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from urllib.request import Request, urlopen
from urllib.error import HTTPError

SHEETS = {
    'ubl25_library':   '18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY',
    'ubl25_documents': '1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg',
}

# ODS XML namespaces
NS_TABLE = '{urn:oasis:names:tc:opendocument:xmlns:table:1.0}'
NS_TEXT = '{urn:oasis:names:tc:opendocument:xmlns:text:1.0}'


def now_iso():
    return datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%S.%fZ')


def export_revision_ods(sheet_id, rev_num, retry_wait=14.0):
    """Download a specific revision as ODS.
    Returns (ods_bytes, http_status, error_msg, hit_429).
    On 429: waits retry_wait seconds then retries (up to 2 retries)."""
    url = (f'https://docs.google.com/spreadsheets/export'
           f'?id={sheet_id}&revision={rev_num}&exportFormat=ods')
    headers = {'Authorization': f'Bearer {TOKEN}'}
    hit_429 = False
    for attempt in range(3):
        try:
            req = Request(url, headers=headers)
            with urlopen(req, timeout=120) as resp:
                data = resp.read()
                if len(data) > 500:
                    return data, 200, None, hit_429
                return None, 200, f'too small ({len(data)} bytes)', hit_429
        except HTTPError as e:
            if e.code == 429 and attempt < 2:
                hit_429 = True
                print(f'429({retry_wait:.0f}s)...', end='', flush=True)
                time.sleep(retry_wait)
                continue
            if e.code in (500, 502, 503) and attempt < 2:
                print(f'retry({e.code}, {retry_wait:.0f}s)...', end='', flush=True)
                time.sleep(retry_wait)
                continue
            return None, e.code, str(e.code), hit_429
        except Exception as exc:
            if attempt < 2:
                time.sleep(retry_wait)
                continue
            return None, 0, str(exc), hit_429
    return None, 0, 'max retries', hit_429


def _extract_cell_text(cell_elem):
    """Extract text from an ODS cell, including nested <text:p>/<text:span>."""
    parts = []
    for p in cell_elem.findall(f'{NS_TEXT}p'):
        if p.text:
            parts.append(p.text)
        for child in p:
            if child.text:
                parts.append(child.text)
            if child.tail:
                parts.append(child.tail)
    return ''.join(parts)


def ods_data_fingerprint(ods_bytes):
    """Compute a stable fingerprint of the actual spreadsheet DATA.

    Extracts cell text from all tables except any sheet whose name
    contains 'log' (case-insensitive). Ignores ODS metadata, styles,
    formulas, and style-name attributes — only looks at visible text.

    Immune to:
    - ODS metadata jitter (timestamps, generator strings)
    - Style name shuffling (ce1 vs ce23)
    - Formula reference shifts from column/row insertions
    - Logs-sheet noise

    Returns SHA256 hex digest, or None on error.
    """
    try:
        with zipfile.ZipFile(io.BytesIO(ods_bytes)) as zf:
            root = ET.fromstring(zf.read('content.xml'))
    except Exception:
        return None

    tables = root.findall(f'.//{NS_TABLE}table')
    parts = []

    for table in tables:
        name = table.get(f'{NS_TABLE}name', '')
        if 'log' in name.lower():
            continue

        for row in table.findall(f'.//{NS_TABLE}table-row'):
            row_repeat = int(row.get(
                f'{NS_TABLE}number-rows-repeated', '1'))
            if row_repeat > 10000:
                continue  # skip empty spacer rows

            row_texts = []
            for cell in row.findall(f'{NS_TABLE}table-cell'):
                col_repeat = int(cell.get(
                    f'{NS_TABLE}number-columns-repeated', '1'))
                if col_repeat > 1000:
                    continue  # skip empty spacer columns

                text = _extract_cell_text(cell)
                # Include empty cells to preserve column alignment
                for _ in range(col_repeat):
                    row_texts.append(text)

            # Only include rows with at least one non-empty cell
            if any(t for t in row_texts):
                row_str = '\t'.join(row_texts)
                for _ in range(row_repeat):
                    parts.append(row_str)

    canonical = '\n'.join(parts)
    return hashlib.sha256(canonical.encode('utf-8')).hexdigest()


def ods_content_hash(ods_bytes):
    """SHA256 of raw content.xml (kept for backward compat in results log)."""
    try:
        with zipfile.ZipFile(io.BytesIO(ods_bytes)) as zf:
            return hashlib.sha256(zf.read('content.xml')).hexdigest()
    except Exception:
        return None


print('Helpers ready')

## Step 3: Establish live reference fingerprint

Download `MAX_REV` (the latest revision). By definition, this IS the
current content. Compute its data fingerprint — this becomes our
"collapsed" detector. No hardcoded hashes needed.

In [ ]:
sheet_id = SHEETS[TEST_SHEET]
ods_dir = DRIVE_DIR / TEST_SHEET
ods_dir.mkdir(exist_ok=True)

print(f'Downloading reference: rev {MAX_REV} (latest = current content)...')
ref_ods, ref_status, ref_err, _ = export_revision_ods(sheet_id, MAX_REV)

if not ref_ods:
    raise RuntimeError(
        f'Cannot download reference rev {MAX_REV}: '
        f'HTTP {ref_status} {ref_err}')

REFERENCE_FP = ods_data_fingerprint(ref_ods)
REFERENCE_SIZE = len(ref_ods)
REFERENCE_CONTENT_HASH = ods_content_hash(ref_ods)

if not REFERENCE_FP:
    raise RuntimeError('Failed to compute reference fingerprint')

print(f'  Fingerprint: {REFERENCE_FP[:16]}...{REFERENCE_FP[-8:]}')
print(f'  Size: {REFERENCE_SIZE:,} bytes')
print(f'  Content hash: {REFERENCE_CONTENT_HASH[:16]}...')
print(f'\nThis fingerprint = \"collapsed to current\". Any download')
print(f'whose data fingerprint matches this is NOT a real historical revision.')

## Step 4: Audit existing Drive files (cached)

Scan all `rev-*.ods.gz` files on Drive. Use a **fingerprint cache** to
avoid re-reading files we've already fingerprinted. Only files not in
the cache (new or replaced) get fingerprinted from disk.

The cache is stored as `audit-cache-{sheet}.json` on Drive and persists
across runs. First run: fingerprints everything (~2-5 min). Subsequent
runs: loads cache, fingerprints only new files (<1 second if nothing changed).

In [ ]:
print(f'Scanning Drive files in {ods_dir}...')

# ── Fingerprint cache ──
# Persists across runs so we don't re-read files we've already seen.
# Only files not in cache (or with different gz size) get fingerprinted.
cache_path = DRIVE_DIR / f'audit-cache-{TEST_SHEET}.json'
fp_cache = {}  # rev_num (int) -> { 'fp': str, 'gz_size': int }

if cache_path.exists():
    try:
        raw = json.loads(cache_path.read_text())
        for k, v in raw.get('revisions', {}).items():
            fp_cache[int(k)] = v
        print(f'  Loaded fingerprint cache: {len(fp_cache)} entries')
    except Exception as e:
        print(f'  Cache load failed ({e}), starting fresh')
        fp_cache = {}
else:
    print(f'  No fingerprint cache — first run, will fingerprint all files')


def save_fp_cache():
    """Write fingerprint cache to Drive."""
    cache_path.write_text(json.dumps({
        'sheet': TEST_SHEET,
        'reference_fp': REFERENCE_FP,
        'timestamp': now_iso(),
        'revisions': {str(k): v for k, v in fp_cache.items()},
    }, indent=2))


# ── Discover which revisions exist on Drive ──
drive_files = sorted(ods_dir.glob('rev-*.ods.gz'))
drive_revs = {}
for p in drive_files:
    m = re.match(r'rev-(\d+)\.ods\.gz$', p.name)
    if m:
        drive_revs[int(m.group(1))] = p

print(f'  Found {len(drive_revs)} files on Drive')

# ── Classify each file: use cache when possible, fingerprint when not ──
collapsed_revs = set()
historical_revs = set()
error_revs = set()

cache_hits = 0
cache_misses = 0
t0 = time.time()

for i, (rev_num, gz_path) in enumerate(sorted(drive_revs.items())):
    gz_size = gz_path.stat().st_size

    # Check cache: hit if we have this rev AND gz size matches
    cached = fp_cache.get(rev_num)
    if cached and cached.get('gz_size') == gz_size:
        fp = cached['fp']
        cache_hits += 1
    else:
        # Cache miss: read and fingerprint the file
        try:
            ods_bytes = gzip.decompress(gz_path.read_bytes())
            fp = ods_data_fingerprint(ods_bytes)
        except Exception:
            fp = None
        cache_misses += 1

        # Update cache
        if fp is not None:
            fp_cache[rev_num] = {'fp': fp, 'gz_size': gz_size}
        else:
            fp_cache.pop(rev_num, None)

        # Progress for slow first run
        if cache_misses % 200 == 0:
            elapsed = time.time() - t0
            print(f'    Fingerprinted {cache_misses} files '
                  f'({elapsed:.0f}s elapsed)...')

    # Classify
    if fp is None:
        error_revs.add(rev_num)
    elif fp == REFERENCE_FP:
        collapsed_revs.add(rev_num)
    else:
        historical_revs.add(rev_num)

# Remove stale cache entries (files deleted from Drive)
stale = set(fp_cache.keys()) - set(drive_revs.keys())
for rev_num in stale:
    del fp_cache[rev_num]

audit_elapsed = time.time() - t0

# Save updated cache
save_fp_cache()

# Find missing revisions
all_revs = set(range(1, MAX_REV + 1))
missing_revs = all_revs - set(drive_revs.keys())

# Revisions that need downloading
need_download = sorted(collapsed_revs | missing_revs | error_revs)

print(f'\nAudit complete in {audit_elapsed:.1f}s:')
print(f'  Cache: {cache_hits} hits, {cache_misses} misses'
      f'{", " + str(len(stale)) + " stale removed" if stale else ""}')
print(f'  Historical (verified):  {len(historical_revs)}')
print(f'  Collapsed (= current):  {len(collapsed_revs)}')
print(f'  Missing (not on Drive): {len(missing_revs)}')
print(f'  Errors (unreadable):    {len(error_revs)}')
print(f'  Need downloading:       {len(need_download)}')

if collapsed_revs:
    sc = sorted(collapsed_revs)
    print(f'\n  Collapsed range: rev {sc[0]} - {sc[-1]}')
    if len(sc) <= 20:
        print(f'  Collapsed revs: {sc}')

if need_download:
    print(f'\n  Download range: rev {need_download[0]} - {need_download[-1]}')
else:
    print(f'\n  All revisions verified! Nothing to download.')

## Step 5: Build download plan + reset checkpoint

Starts `RESUME_MARGIN` verified-good revisions before the first
revision that needs downloading. This verifies the API is returning
genuine data before hitting the problem zone.

Resets the checkpoint and cleans results from the resume point onward.

In [ ]:
results_path = DRIVE_DIR / f'slow-validation-{TEST_SHEET}.json'
checkpoint_path = DRIVE_DIR / f'checkpoint-{TEST_SHEET}.json'

if not need_download:
    print('Nothing to download — skipping plan/reset.')
    RESUME_FROM = None
else:
    first_needed = need_download[0]
    RESUME_FROM = max(1, first_needed - RESUME_MARGIN)

    print(f'Download plan:')
    print(f'  First revision that needs downloading: rev {first_needed}')
    print(f'  Resume from: rev {RESUME_FROM} '
          f'({first_needed - RESUME_FROM} margin revisions for verification)')
    print(f'  Total revisions to process: {MAX_REV - RESUME_FROM + 1} '
          f'(rev {RESUME_FROM} -> {MAX_REV})')

    n_batches = (MAX_REV - RESUME_FROM + 1 + BATCH_SIZE - 1) // BATCH_SIZE
    est_minutes = ((MAX_REV - RESUME_FROM + 1) * REQUEST_DELAY / 60 +
                   (n_batches - 1) * COOLDOWN_MINUTES)
    print(f'  Estimated runtime: ~{est_minutes:.0f} min ({est_minutes/60:.1f} hrs)')

    # ── Clean up results from RESUME_FROM onward ──
    if results_path.exists():
        results_data = json.loads(results_path.read_text())
        tests_before = results_data.get('tests', [])
        n_before = len(tests_before)

        tests_after = [t for t in tests_before if t['rev'] < RESUME_FROM]
        n_removed = n_before - len(tests_after)
        results_data['tests'] = tests_after

        for key in ['canary_triggered', 'canary_at_rev', 'canary_after_n',
                    'completed', 'adaptive_backoff']:
            results_data.pop(key, None)

        results_data['summary'] = {
            'total_tested': len(tests_after),
            'total_ok': sum(1 for t in tests_after if t['status'] == 'ok'),
            'total_error': sum(1 for t in tests_after
                              if t['status'] == 'error'),
            'historical': sum(1 for t in tests_after
                             if t.get('is_historical')),
            'current': sum(1 for t in tests_after if t.get('is_current')),
            'replaced_on_drive': sum(1 for t in tests_after
                                    if t.get('replaced_on_drive')),
            'canary_triggered': False,
        }
        results_data['last_updated'] = now_iso()
        results_path.write_text(json.dumps(results_data, indent=2))

        if n_removed > 0:
            print(f'\n  Cleaned results: {n_before} -> {len(tests_after)} '
                  f'({n_removed} removed, rev >= {RESUME_FROM})')
        else:
            print(f'\n  Results: {len(tests_after)} entries (nothing to remove)')
    else:
        print(f'\n  No previous results file.')

    # ── Set checkpoint ──
    resume_rev = RESUME_FROM - 1
    checkpoint_path.write_text(json.dumps({
        'sheet': TEST_SHEET,
        'last_rev': resume_rev,
        'delay': REQUEST_DELAY,
        'speedup_threshold': 12,
        'ok_streak': 0,
        'total_429s': 0,
        'timestamp': now_iso(),
    }, indent=2))
    print(f'  Checkpoint: reset to rev {resume_rev}')
    print(f'\nReady! Step 6 will download from rev {RESUME_FROM} to {MAX_REV}.')

## Step 6: Download loop

### Collapse detection
Each download's data fingerprint is compared against the live reference.
Match = collapsed. No match = genuine historical data.

### Canary
Stops if `CANARY_THRESHOLD` consecutive downloads have the same
data fingerprint.

### Batch cooldown
After every `BATCH_SIZE` downloads, pauses for `COOLDOWN_MINUTES`.

### Adaptive rate
- **429 retry**: waits `current_delay + 4s`
- **After 429**: base delay **+1s**, speedup threshold **+1**
- **After N OK**: decrease delay by **0.5s** (min 10s)

### Saves to Drive
- Only genuinely historical data (fingerprint != reference)
- Replaces collapsed files when historical data is recovered
- Checkpoint written after every request

In [ ]:
if RESUME_FROM is None:
    print('Nothing to download. All revisions verified!')
else:
    # ── Build test plan ──
    TEST_PLAN = list(range(RESUME_FROM, MAX_REV + 1))

    # ── Adaptive backoff state ──
    MIN_DELAY = 10.0
    RETRY_GAP = 4.0
    current_delay = REQUEST_DELAY
    ok_streak = 0
    speedup_threshold = 12
    total_429s = 0
    rate_changes = []
    resume_after_rev = None

    # ── Batch cooldown state ──
    batch_count = 0
    cooldown_seconds = COOLDOWN_MINUTES * 60
    total_cooldowns = 0

    # ── Reference refresh state ──
    downloads_since_refresh = 0
    reference_fp = REFERENCE_FP

    # ── Load checkpoint ──
    if checkpoint_path.exists():
        cp = json.loads(checkpoint_path.read_text())
        resume_after_rev = cp.get('last_rev')
        current_delay = max(cp.get('delay', current_delay), MIN_DELAY)
        speedup_threshold = cp.get('speedup_threshold', speedup_threshold)
        ok_streak = cp.get('ok_streak', 0)
        total_429s = cp.get('total_429s', 0)
        print(f'Checkpoint: resume after rev {resume_after_rev}, '
              f'delay={current_delay:.1f}s, threshold={speedup_threshold}')
    else:
        print('No checkpoint — starting fresh')


    def save_checkpoint(rev_num):
        checkpoint_path.write_text(json.dumps({
            'sheet': TEST_SHEET,
            'last_rev': rev_num,
            'delay': round(current_delay, 1),
            'speedup_threshold': speedup_threshold,
            'ok_streak': ok_streak,
            'total_429s': total_429s,
            'timestamp': now_iso(),
        }, indent=2))


    # ── Load or init results ──
    if results_path.exists():
        results = json.loads(results_path.read_text())
        done_revs = {r['rev'] for r in results.get('tests', [])}
        print(f'Results: {len(done_revs)} revisions already logged')
    else:
        results = {
            'sheet_key': TEST_SHEET,
            'sheet_id': sheet_id,
            'config': {
                'request_delay': REQUEST_DELAY,
                'canary_threshold': CANARY_THRESHOLD,
                'batch_size': BATCH_SIZE,
                'cooldown_minutes': COOLDOWN_MINUTES,
            },
            'reference_fingerprint': reference_fp,
            'started': now_iso(),
            'tests': [],
            'summary': {},
        }
        done_revs = set()


    def save_results():
        results['last_updated'] = now_iso()
        results['reference_fingerprint'] = reference_fp
        tests = results['tests']
        results['summary'] = {
            'total_tested': len(tests),
            'total_ok': sum(1 for t in tests if t['status'] == 'ok'),
            'total_error': sum(1 for t in tests if t['status'] == 'error'),
            'historical': sum(1 for t in tests if t.get('is_historical')),
            'current': sum(1 for t in tests if t.get('is_current')),
            'replaced_on_drive': sum(1 for t in tests
                                    if t.get('replaced_on_drive')),
            'canary_triggered': results.get('canary_triggered', False),
            'total_cooldowns': total_cooldowns,
        }
        results_path.write_text(json.dumps(results, indent=2))


    def log_rate_change(reason, old_delay, new_delay):
        arrow = '\u25b2' if new_delay > old_delay else '\u25bc'
        print(f'  {arrow} RATE {old_delay:.1f}s -> {new_delay:.1f}s  '
              f'(retry={new_delay + RETRY_GAP:.0f}s)  ({reason})')
        rate_changes.append({
            'timestamp': now_iso(),
            'reason': reason,
            'old_delay': old_delay,
            'new_delay': new_delay,
        })


    def do_cooldown(reason):
        global total_cooldowns, batch_count
        total_cooldowns += 1
        batch_count = 0
        mins = cooldown_seconds / 60
        print(f'\n  {"="*60}')
        print(f'  COOLDOWN #{total_cooldowns}: {reason}')
        print(f'  Pausing {mins:.0f} minutes...')
        print(f'  {"="*60}')
        remaining = cooldown_seconds
        while remaining > 0:
            wait = min(60, remaining)
            time.sleep(wait)
            remaining -= wait
            if remaining > 0:
                print(f'  ... {remaining/60:.0f} min remaining', flush=True)
        print(f'  Cooldown complete.\n')


    def refresh_reference():
        """Re-download MAX_REV to update the reference fingerprint."""
        global reference_fp, downloads_since_refresh
        print(f'\n  Refreshing reference (downloading rev {MAX_REV})...',
              end='', flush=True)
        ref_ods, status, err, _ = export_revision_ods(sheet_id, MAX_REV)
        if ref_ods:
            new_fp = ods_data_fingerprint(ref_ods)
            if new_fp and new_fp != reference_fp:
                print(f' UPDATED: {reference_fp[:12]}... -> {new_fp[:12]}...')
                reference_fp = new_fp
            else:
                print(f' unchanged ({reference_fp[:12]}...)')
        else:
            print(f' FAILED (HTTP {status}), keeping old reference')
        downloads_since_refresh = 0


    # ── Skip logic ──
    skip_up_to = resume_after_rev is not None
    revs_skipped = 0

    # ── Cache save counter (save every N writes to avoid Drive thrashing) ──
    cache_writes_pending = 0
    CACHE_SAVE_INTERVAL = 50

    # ── Main loop ──
    print(f'\n{"="*70}')
    print(f'Starting: {TEST_SHEET} (rev {RESUME_FROM} -> {MAX_REV})')
    print(f'  delay={current_delay:.1f}s, '
          f'retry_wait={current_delay + RETRY_GAP:.0f}s, '
          f'speedup_after={speedup_threshold}')
    print(f'  Reference FP: {reference_fp[:16]}...')
    print(f'  Batch: {BATCH_SIZE}, cooldown: {COOLDOWN_MINUTES}min')
    print(f'{"="*70}\n')

    consecutive_same_fp = 0
    last_fp = None
    canary_triggered = False

    for i, rev_num in enumerate(TEST_PLAN):
        # ── Skip: checkpoint-based resume ──
        if skip_up_to:
            if rev_num == resume_after_rev:
                skip_up_to = False
            revs_skipped += 1
            continue

        # ── Skip: already in results ──
        if rev_num in done_revs:
            continue

        # ── Batch cooldown ──
        if batch_count > 0 and batch_count % BATCH_SIZE == 0:
            do_cooldown(f'completed batch of {BATCH_SIZE} revisions')
            consecutive_same_fp = 0
            last_fp = None

        # ── Periodic reference refresh ──
        if downloads_since_refresh >= REFERENCE_REFRESH_INTERVAL:
            refresh_reference()
            time.sleep(current_delay)

        # ── Download ──
        t0 = time.time()
        retry_wait = current_delay + RETRY_GAP
        print(f'  [{i+1}/{len(TEST_PLAN)}] rev-{rev_num:>5}: ',
              end='', flush=True)

        ods_data, http_status, error, hit_429 = export_revision_ods(
            sheet_id, rev_num, retry_wait=retry_wait)
        dl_elapsed = time.time() - t0

        # ── 429 handling ──
        if hit_429:
            total_429s += 1
            ok_streak = 0
            old_delay = current_delay
            current_delay += 1.0
            speedup_threshold += 1
            if current_delay != old_delay:
                log_rate_change(
                    f'429 #{total_429s}, thr->{speedup_threshold}',
                    old_delay, current_delay)

        if not ods_data:
            print(f'ERROR (HTTP {http_status}: {error}) [{dl_elapsed:.1f}s]')
            results['tests'].append({
                'rev': rev_num,
                'status': 'error',
                'http_status': http_status,
                'error': error,
                'hit_429': hit_429,
                'delay_at_time': current_delay,
                'timestamp': now_iso(),
                'elapsed': round(dl_elapsed, 2),
            })
            consecutive_same_fp = 0
            last_fp = None
            save_results()
            save_checkpoint(rev_num)
            time.sleep(current_delay)
            continue

        # ── Speedup probe ──
        if not hit_429:
            ok_streak += 1
            if ok_streak >= speedup_threshold:
                old_delay = current_delay
                current_delay = max(current_delay - 0.5, MIN_DELAY)
                ok_streak = 0
                speedup_threshold = max(speedup_threshold - 1, 6)
                if current_delay != old_delay:
                    log_rate_change(
                        f'{speedup_threshold + 1} OK, next after '
                        f'{speedup_threshold}',
                        old_delay, current_delay)

        # ── Fingerprint ──
        data_fp = ods_data_fingerprint(ods_data)
        content_hash = ods_content_hash(ods_data)
        ods_size = len(ods_data)

        is_current = (data_fp == reference_fp)
        is_historical = (not is_current and data_fp is not None)

        # ── Drive state for this rev ──
        drive_is_collapsed = (rev_num in collapsed_revs)
        drive_exists = (rev_num in drive_revs)

        # ── Canary tracking ──
        if data_fp == last_fp:
            consecutive_same_fp += 1
        else:
            consecutive_same_fp = 1
            last_fp = data_fp

        # ── Result entry ──
        entry = {
            'rev': rev_num,
            'status': 'ok',
            'content_hash': content_hash,
            'data_fingerprint': data_fp,
            'ods_size': ods_size,
            'is_current': is_current,
            'is_historical': is_historical,
            'drive_was_collapsed': drive_is_collapsed,
            'consecutive_same_fp': consecutive_same_fp,
            'hit_429': hit_429,
            'delay_at_time': current_delay,
            'timestamp': now_iso(),
            'elapsed': round(dl_elapsed, 2),
            'replaced_on_drive': False,
        }

        # ── Status line ──
        parts = [f'{ods_size:,}b']
        if is_current:
            parts.append('COLLAPSED')
        else:
            parts.append(f'HISTORICAL ({data_fp[:12]}...)')

        if drive_is_collapsed and is_historical:
            parts.append('RECOVERED!')
        elif not drive_exists:
            parts.append('NEW')

        parts.append(f'[{dl_elapsed:.1f}s]')
        if consecutive_same_fp > 1:
            parts.append(f'(same x{consecutive_same_fp})')
        parts.append(f'd={current_delay:.1f}s')
        parts.append(f'ok={ok_streak}/{speedup_threshold}')
        parts.append(f'B={batch_count+1}/{BATCH_SIZE}')
        print(' '.join(parts))

        # ── Save to Drive ──
        if is_historical:
            gz_path = ods_dir / f'rev-{rev_num}.ods.gz'
            should_write = False

            if drive_is_collapsed:
                should_write = True
                entry['replaced_on_drive'] = True
                print(f'         >>> REPLACING collapsed file on Drive')
            elif not gz_path.exists():
                should_write = True

            if should_write:
                gz_data = gzip.compress(ods_data, compresslevel=6)
                gz_path.write_bytes(gz_data)
                collapsed_revs.discard(rev_num)
                historical_revs.add(rev_num)
                missing_revs.discard(rev_num)
                # Update fingerprint cache — we know the fp, no re-read needed
                fp_cache[rev_num] = {'fp': data_fp, 'gz_size': len(gz_data)}
                cache_writes_pending += 1

        results['tests'].append(entry)
        done_revs.add(rev_num)
        batch_count += 1
        downloads_since_refresh += 1
        save_results()
        save_checkpoint(rev_num)

        # ── Periodic cache save (avoid writing every single iteration) ──
        if cache_writes_pending >= CACHE_SAVE_INTERVAL:
            save_fp_cache()
            cache_writes_pending = 0

        # ── Canary ──
        if consecutive_same_fp >= CANARY_THRESHOLD:
            tag = 'COLLAPSED' if is_current else 'IDENTICAL'
            print(f'\n  *** CANARY: {consecutive_same_fp} identical '
                  f'fingerprints in a row ***')
            print(f'  *** FP: {data_fp[:32]}... ({tag})')
            print(f'  *** Stopping. Re-run later to try again.')
            results['canary_triggered'] = True
            results['canary_at_rev'] = rev_num
            results['canary_after_n'] = i + 1
            save_results()
            canary_triggered = True
            break

        # ── Wait ──
        remaining_delay = max(0, current_delay - (time.time() - t0))
        if remaining_delay > 0:
            time.sleep(remaining_delay)

    # ── Final save (results + cache) ──
    results['completed'] = now_iso()
    results['adaptive_backoff'] = {
        'final_delay': current_delay,
        'total_429s': total_429s,
        'final_speedup_threshold': speedup_threshold,
        'rate_changes': rate_changes,
        'total_cooldowns': total_cooldowns,
    }
    save_results()
    save_fp_cache()  # persist any remaining cache updates

    if revs_skipped:
        print(f'\n  Skipped {revs_skipped} revisions '
              f'(checkpoint: rev {resume_after_rev})')
    if not canary_triggered:
        print(f'\n  Complete! All revisions processed.')
    print(f'  Cooldowns: {total_cooldowns}')
    print(f'  Final delay: {current_delay:.1f}s, 429s: {total_429s}')

## Step 7: Results

In [ ]:
results = json.loads(results_path.read_text())
tests = results['tests']

print(f'{"="*70}')
print(f'RESULTS: {TEST_SHEET}')
print(f'{"="*70}')
print(f'  Started:   {results.get("started", "?")}')
print(f'  Completed: {results.get("completed", "?")}')
print(f'  Reference: {results.get("reference_fingerprint", "?")[:24]}...')
print()

s = results.get('summary', {})
print(f'  Total tested:       {s.get("total_tested", 0)}')
print(f'  OK:                 {s.get("total_ok", 0)}')
print(f'  Errors:             {s.get("total_error", 0)}')
print(f'  Historical:         {s.get("historical", 0)}')
print(f'  Collapsed:          {s.get("current", 0)}')
print(f'  Replaced on Drive:  {s.get("replaced_on_drive", 0)}')
print(f'  Canary triggered:   {s.get("canary_triggered", False)}')

ab = results.get('adaptive_backoff', {})
if ab:
    print(f'\n  Final delay: {ab.get("final_delay", "?")}s, '
          f'429s: {ab.get("total_429s", 0)}, '
          f'cooldowns: {ab.get("total_cooldowns", 0)}')

# Collapse blocks
collapsed_tests = [t for t in tests
                   if t.get('is_current') and t['status'] == 'ok']
if collapsed_tests:
    print(f'\n  COLLAPSED downloads ({len(collapsed_tests)}):')
    runs = []
    run_start = collapsed_tests[0]['rev']
    prev_rev = run_start
    for t in collapsed_tests[1:]:
        if t['rev'] == prev_rev + 1:
            prev_rev = t['rev']
        else:
            runs.append((run_start, prev_rev))
            run_start = t['rev']
            prev_rev = t['rev']
    runs.append((run_start, prev_rev))
    for start, end in runs:
        count = end - start + 1
        print(f'    rev {start}-{end} ({count})')
else:
    print(f'\n  No collapsed downloads!')

# Recovered
recovered = [t for t in tests if t.get('replaced_on_drive')]
if recovered:
    print(f'\n  RECOVERED ({len(recovered)} files replaced on Drive)')

# Overall Drive state
print(f'\n  Drive state:')
print(f'    Historical:  {len(historical_revs)}')
print(f'    Collapsed:   {len(collapsed_revs)}')
print(f'    Missing:     {len(missing_revs)}')
pct = 100 * len(historical_revs) / MAX_REV
print(f'    Coverage:    {len(historical_revs)}/{MAX_REV} ({pct:.1f}%)')

---

## For Claude: How to check progress

**IMPORTANT:** Never use WebFetch — it caches results (15 min).
Always use `curl` or `python3 urllib` via the Bash tool.

### Key IDs

| What | ID |
|------|-----|
| **Public root folder** | `1SVXV_8CF4ib9YsVZ6G7AqIP3gNK6q3Gj` |
| Library ODS subfolder | `1JRvRaqsNP_G-9xICwrgGpY-ArwV8ej1w` |
| Documents ODS subfolder | `1DsufM2yMqcbE8kR-RH1i5rSBcvPivOxA` |

### Quick check

```bash
# List files
python3 work-sheets/scripts/explore-public-drive.py \
    1SVXV_8CF4ib9YsVZ6G7AqIP3gNK6q3Gj /tmp/drive-listing.json

# Download checkpoint by file ID
curl -sL "https://drive.google.com/uc?export=download&id={FILE_ID}" | python3 -m json.tool
```

### Done when

- Library: `last_rev` = 2005
- Documents: `last_rev` = 2204